In [ ]:
%pip install -q kagglehub openpyxl
import kagglehub, os, glob
path = kagglehub.dataset_download("teertha/personal-loan-modeling")
csv = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
DATA_PATH = csv[0]
print("Using:", DATA_PATH)

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv(DATA_PATH)

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df=df.drop('ID',axis=1)

In [ ]:
df.info()

In [ ]:
df=df.drop('ZIP Code',axis=1)

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split

X=df.drop('Personal Loan',axis=1)
y=df['Personal Loan']
X_train,X_test,y_train,y_test=train_test_split(X,y, test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
model=Sequential(
    [
        Dense(16,input_dim=X_train.shape[1], activation='relu'),
        Dense(8,activation='relu'),
        Dense(1,activation='sigmoid')
    ]
)

In [ ]:
model.compile(optimizer='Adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.fit(X_train,y_train,epochs=50,batch_size=32,validation_split=0.1)

In [ ]:
model1=Sequential(
    [
        Dense(16,input_dim=X_train.shape[1], activation='sigmoid'),
        Dense(8,activation='sigmoid'),
        Dense(1,activation='sigmoid')
    ]
)
model1.compile(optimizer='Adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model1.fit(X_train,y_train,epochs=50,batch_size=32,validation_split=0.1)

In [ ]:
model2=Sequential(
    [
        Dense(16,input_dim=X_train.shape[1], activation='tanh'),
        Dense(8,activation='tanh'),
        Dense(1,activation='sigmoid')
    ]
)
model2.compile(optimizer='Adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model2.fit(X_train,y_train,epochs=50,batch_size=32,validation_split=0.1)

In [ ]:
history1 = model1.fit(X_train, y_train,
                      validation_data=(X_test, y_test),
                      epochs=50,
                      batch_size=32,
                      verbose=0)

history2 = model2.fit(X_train, y_train,
                      validation_data=(X_test, y_test),
                      epochs=50,
                      batch_size=32,
                      verbose=0)

history3 = model.fit(X_train, y_train,
                      validation_data=(X_test, y_test),
                      epochs=50,
                      batch_size=32,
                      verbose=0)

In [ ]:
plt.plot(history1.history['val_loss'], label='Sigmoid')
plt.plot(history2.history['val_loss'], label='Tanh')
plt.plot(history3.history['val_loss'], label='Relu')

plt.xlabel('Epochs')
plt.ylabel('Validation Loss (Cost)')
plt.legend()
plt.show()

In [ ]:
loss1, acc1 = model1.evaluate(X_test, y_test, verbose=0)
loss2, acc2 = model2.evaluate(X_test, y_test, verbose=0)
loss3, acc3 = model.evaluate(X_test, y_test, verbose=0)

print(acc1,acc2,acc3)

In [ ]:
from sklearn.metrics import f1_score

y_pred1 = (model1.predict(X_test) > 0.5).astype(int)
y_pred2 = (model2.predict(X_test) > 0.5).astype(int)
y_pred3 = (model.predict(X_test) > 0.5).astype(int)

In [ ]:
print("Sigmoid F1-score:", f1_score(y_test, y_pred1))
print("tanh F1-score:", f1_score(y_test, y_pred2))
print("relu F1-score:", f1_score(y_test, y_pred3))